In [7]:
%load_ext autoreload
%autoreload 2

In [17]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

import config as cnf 
import utils as utl

In [18]:
df_train                = pd.read_csv(cnf.path['train_path'])
df_test                 = pd.read_csv(cnf.path['test_path'])
df_gender_submission    = pd.read_csv(cnf.path['gender_submission_path'])

### Логистическая регрессия

In [ ]:
X, y = utl.predobr(df_train, df_test, df_gender_submission)

combo_param = utl.parametrs(cnf.RandomForestParam) + utl.parametrs(cnf.LogRegParam)
combo_param += utl.parametrs(cnf.KNNParam)
combo_param += utl.parametrs(cnf.DecisionTreeParam)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cnf.random_state)
results = []

for combo in combo_param:
    name_model = combo['name_model']
    
    if name_model == 'RandomForest':

        n_estimators = combo['n_estimators']
        max_depth = combo['max_depth']
        min_samples_leaf = combo['min_samples_leaf']

        print("Количество деревьев: ", n_estimators, " | Максимальная глубина: ", max_depth, " | Минимум на листе :", min_samples_leaf)

        array_train_accuracy = []
        array_test_accuracy = []

        for i, (train_index, test_index) in enumerate(skf.split(X, y)):

            model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=cnf.random_state, min_samples_leaf=min_samples_leaf)
            train_accuracy, test_accuracy = utl.fit_model(model, X, y, train_index, test_index, scaler=False)
            array_train_accuracy.append(train_accuracy)
            array_test_accuracy.append(test_accuracy)
            
        mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = utl.mean_accuracy(array_train_accuracy, array_test_accuracy)

        results.append({
            'name_model'            :   name_model
            , 'n_estimator'         :   n_estimators
            , 'max_depth'           :   max_depth
            , 'min_samples_leaf'    :   min_samples_leaf
            , 'train_accuracy'      :   mean_train_accuracy
            , 'train_std'           :   mean_train_std
            , 'test_accuracy'       :   mean_test_accuracy
            , 'test_std'            :   mean_test_std
        })
        
    elif name_model == 'DecisionTree':
        
        max_depth = combo['max_depth']
        min_samples_leaf = combo['min_samples_leaf']

        print("Максимальная глубина: ", max_depth, " | Минимум на листе :", min_samples_leaf)

        array_train_accuracy = []
        array_test_accuracy = []

        for i, (train_index, test_index) in enumerate(skf.split(X, y)):

            model = DecisionTreeClassifier(max_depth=max_depth, random_state=cnf.random_state, min_samples_leaf=min_samples_leaf)
            train_accuracy, test_accuracy = utl.fit_model(model, X, y, train_index, test_index, scaler=False)
            array_train_accuracy.append(train_accuracy)
            array_test_accuracy.append(test_accuracy)
            
        mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = utl.mean_accuracy(array_train_accuracy, array_test_accuracy)

        results.append({
            'name_model'            :   name_model
            , 'max_depth'           :   max_depth
            , 'min_samples_leaf'    :   min_samples_leaf
            , 'train_accuracy'      :   mean_train_accuracy
            , 'train_std'           :   mean_train_std
            , 'test_accuracy'       :   mean_test_accuracy
            , 'test_std'            :   mean_test_std
        })        
    
    elif name_model == 'LogReg':
        
        print("LogReg")
        array_train_accuracy = []
        array_test_accuracy = []
        for i, (train_index, test_index) in enumerate(skf.split(X, y)):

            model = LogisticRegression(random_state=cnf.random_state)
            train_accuracy, test_accuracy = utl.fit_model(model, X, y, train_index, test_index, scaler=True)
            
            array_train_accuracy.append(train_accuracy)
            array_test_accuracy.append(test_accuracy)
            
        mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = utl.mean_accuracy(array_train_accuracy, array_test_accuracy)

        results.append({
            'name_model'            :   name_model
            , 'train_accuracy'      :   mean_train_accuracy
            , 'train_std'           :   mean_train_std
            , 'test_accuracy'       :   mean_test_accuracy
            , 'test_std'            :   mean_test_std
        })
        
    elif name_model == 'KNN':

        n_neighbors = combo['n_neighbors']
        
        print("KNN n_neighbors: ", n_neighbors)
        array_train_accuracy = []
        array_test_accuracy = []
        for i, (train_index, test_index) in enumerate(skf.split(X, y)):

            model = KNeighborsClassifier(n_neighbors=n_neighbors)
            train_accuracy, test_accuracy = utl.fit_model(model, X, y, train_index, test_index, scaler=True)
            
            array_train_accuracy.append(train_accuracy)
            array_test_accuracy.append(test_accuracy)            
            
        mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = utl.mean_accuracy(array_train_accuracy, array_test_accuracy)
        results.append({
            'name_model'            :   name_model
            , 'n_neighbors'         :   n_neighbors
            , 'train_accuracy'      :   mean_train_accuracy
            , 'train_std'           :   mean_train_std
            , 'test_accuracy'       :   mean_test_accuracy
            , 'test_std'            :   mean_test_std
        })

NameError: name 'random_state' is not defined

### Случайный лес

In [15]:
results = pd.DataFrame(results)

In [16]:
results['differenc'] = results['train_accuracy'] - results['test_accuracy']
results.sort_values('test_accuracy', ascending=False).head(50)

,name_model,n_estimator,max_depth,min_samples_leaf,train_accuracy,train_std,test_accuracy,test_std,n_neighbors,differenc
3,RandomForest,100.0,9.0,1.0,0.932659,0.004454,0.842854,0.012525,NaN,0.089805
5,RandomForest,100.0,9.0,3.0,0.888046,0.004415,0.839483,0.019818,NaN,0.048563
1,RandomForest,100.0,8.0,2.0,0.894780,0.004647,0.837236,0.022066,NaN,0.057544
0,RandomForest,100.0,8.0,1.0,0.915823,0.006486,0.836125,0.014097,NaN,0.079698
2,RandomForest,100.0,8.0,3.0,0.883558,0.005809,0.834995,0.020443,NaN,0.048563
4,RandomForest,100.0,9.0,2.0,0.905162,0.002320,0.832754,0.018059,NaN,0.072408
19,DecisionTree,NaN,8.0,3.0,0.879064,0.008909,0.829383,0.015938,NaN,0.049681
22,DecisionTree,NaN,9.0,3.0,0.887763,0.006209,0.827142,0.018064,NaN,0.060621
18,DecisionTree,NaN,8.0,2.0,0.883555,0.006451,0.823784,0.012735,NaN,0.059771
17,DecisionTree,NaN,8.0,1.0,0.902074,0.005327,0.821549,0.020218,NaN,0.080525
